# 🌅 Gap and go — opening-range breaks on overnight gaps, and the case for trading them backwards

**The idea.** An overnight gap on elevated opening participation tends to *continue* intraday (gap-and-go); the break of the first-`or_minutes` opening range is the trigger that separates continuation from gap-and-fade. Every other 1m strategy in this repo blacklists the first minutes of the session (`skip_open`) — this one trades **only** a morning window just after the open, the untraded region where liquid names make their biggest, fastest moves. The honest catch: opening-range breakout is Crabel-lineage folk wisdom with thin formal support, gap-continuation evidence is mixed, and the open is where spreads are widest — so the global cost model flatters this strategy more than any sibling. Full thesis in [README.md](./README.md).

**The experiment — and what v1 already answered.** The sweep grid (§5) *is* the ablation, and it now has four axes: `polarity` (continuation vs fade), `entry_mode` (`time` buys every qualifying gap at bar K with no breakout trigger), and each gate on/off. `polarity: fade` is not decoration. §5b's event study — the same signal measured directly, market-demeaned, before any backtest machinery touches it — says the continuation leg **loses**: −11.8 bps of excess return at `gap_min ≥ 2%` over 60 bars (t = −3.3), worse as the gap grows, and stable when entry is delayed 1–6 bars, so it is not bid-ask bounce. Meanwhile the *gate-free* morning breakout is mildly positive. The gap gate is what flips the sign. That is in-sample evidence about a **signal**; only §11 (holdout) at §12's stressed costs is evidence about a **strategy** — which is why both polarities are searched rather than one being assumed.

**The universe.** `Morning200` — 200 names ranked **point-in-time**: eligible on having ≥ 200 sessions of 1m data inside the backtest window, ordered by median daily dollar volume over `2025-01-01 → 2025-06-30` *only*, a window that closes before the backtest opens (built in [`01_universe_screener.ipynb`](../../notebooks/01_universe_screener.ipynb) §8). It replaced `HighOctaneTop20`, which was pinned in 2026-07 by trailing-90-day dollar volume — i.e. selected using data from inside and after this very window. Equal-weight buy-and-hold of that basket returned **+460%/yr** over the train window (SNDK +2255%, BE +1123%, neither with pre-window daily history), so every v1 trial scored `excess_cagr ≈ −4.6` and the whole Pareto front was noise. Here the hurdle is a real ~+47%/yr. One bias survives and has to be said out loud: the 1-minute **backfill list itself** was assembled in 2026, so pool *membership* is still survivorship-tinged — only the ranking within it is point-in-time.

**Scaling to 200 names.** Ten to twenty candidates now fire in the same five minutes against one $10k wallet. `max_candidates` fixes what would otherwise be alphabetical luck: gate-passing names are ranked cross-sectionally at bar K−1 on frozen information (the pre-open watchlist a real scanner builds), only the top few may trade that session, and each entry deploys `1/max_candidates` of the wallet. Adding symbols then improves *selection* instead of multiplying churn.

**Train vs holdout.** The `data:` block sets `end` to a deliberate **train cutoff** — the sweep and the optimizer only ever see data up to it. §11 replays the picked parameters on the untouched ~2-month tail, the one number in this notebook the optimizer could not have fit. Every knob lives in [`config.yaml`](./config.yaml); this notebook only *drives* the shared engine: **nothing here defines behavior**.

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import backtest as bt, jobs, optimize as opt, strategies
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple import plots

strat = strategies.load("gap_and_go")
print("data :", strat.data)
print("params:", strat.params)

In [ ]:
d = strat.data
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]),
                         d["bar"], d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"panel: {panel.close.shape[0]:,} bars x {len(panel.symbols)} symbols "
      f"({panel.close.index[0]:%Y-%m-%d} → {panel.close.index[-1]:%Y-%m-%d})")

## 2 · Strategy overview — hyperparameters and search ranges

What each knob does, before any number crunching:

| param | role in the strategy | intuition |
|---|---|---|
| `or_minutes` | opening-range length (bars) | the range that defines "the open"; frozen at bar K−1 |
| `entry_span` | bars after the range closes in which a signal may fire | window = `[K, K+entry_span]`. An **offset**, not an absolute bar, so a window ending before its own range closes cannot be drawn at all |
| `gap_min` | min \|overnight gap\| vs prior close | **the** thesis knob; 0 = any-breakout control |
| `vol_mult` | first-K volume vs time-of-day expectation | baseline already expects the open surge — this asks "elevated even for the open"; 0 = off |
| `entry_mode` | break / time | control: `time` buys the gap at bar K with no breakout trigger |
| `polarity` | go / fade | **the headline experiment**: same event, opposite position. `fade` buys the gap-down break and shorts the gap-up break |
| `rank_by` / `max_candidates` | how the morning watchlist is ordered, and how many seats it has | the scaling knob. Ranked at bar K−1 on frozen info; each entry deploys `1/max_candidates` of the wallet |
| `min_price` / `min_or_dollar_volume` | per-session tradability floors | a name that is not actually trading this morning is not tradable at 2 bps, whatever its universe rank |
| `atr_window` | ATR smoothing (bars) | day-1 bars include the overnight jump → stops auto-widen on gap mornings |
| `sl_mult` / `sl_trail` | stop in ATRs, trailing or fixed | trail = let the move run; fixed = classic bracket |
| `tp_ratio` | target as a **multiple of the stop** | 0 = no target; a ratio, so a degenerate short-gamma bracket is unrepresentable |
| `horizon` | time stop (bars held) | how long to wait for a move that isn't coming |
| `direction` | long / short / both | applied **after** `polarity`, so a long-only `fade` run trades gap-down bounces |
| `eod_exit` | flat at session close | ALWAYS on: overnight risk is the entry signal, never the hold |

`entry_span` and `tp_ratio` are deliberately relative rather than absolute.
In v1 they were `entry_until` and `tp_mult`, drawn independently of the
parameters they had to stay coherent with, and **17% of that study's trials
were rejected by `valid()` before a single backtest ran**. Making the illegal
region unrepresentable is strictly better than rejecting draws from it.

The table below shows, per parameter: the **default** used by the single
backtest (§3), the hand-picked **sweep grid** (§5), and the **Optuna search
range** (§6). A `—` means the stage keeps the default.

In [ ]:
import pandas as pd

def _fmt_space(spec):
    """One readable line for an optuna.space entry."""
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "default": pd.Series(strat.params),
    "sweep grid (§5)": pd.Series({k: ", ".join(map(str, v)) for k, v in strat.sweep.items()}),
    "optuna search range (§6)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
}).fillna("—")

## 3 · Single backtest with the config defaults

Engine rules (identical for every strategy): signals execute at the **next
bar's adjusted open**, and realistic costs are always charged.

> **Capital model:** ONE shared \$10k wallet (engine cash-shared mode). Here the per-entry fraction is not a free parameter — it is `1 / max_candidates`, derived from the watchlist size, so a full watchlist is a fully invested book and never more. The repo-wide comparability rule; see `notebooks/04_compare.ipynb` for the cross-strategy scoreboard.

In [ ]:
import pandas as pd
pf = bt.run_strategy(strat, panel, settings=settings)
stats = bt.portfolio_stats(pf)
fees, slippage = bt.panel_costs(settings.costs, panel)
stats |= bt.active_stats(pf, panel, fees=fees, slippage=slippage,
                         total_return=stats["total_return"])
pd.Series(stats).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity, config defaults").show()

## 4 · Inspect a symbol — trades and opening-range overlays

`inspect_symbol` is the microscope of this notebook, used twice: here on the
most-traded symbol with the *default* parameters, and again in §10 on the
*optimized* ones. How to read the figure:

* **Price panel** — the frozen opening-range high/low (drawn from the true
  high/low, the same series the signal uses) and the prior session's close the
  gap is measured from. ▲ blue = long entry; ▼ orange = short entry; ✕ = exit
  (the ATR stop — trailing by default — the `horizon` time stop, or the
  mandatory session-close flat). The entry→exit segment is green when the
  trade won, red when it lost. Which side of a break is long and which is
  short depends on `polarity`: under `go` a gap-up range-high break is a buy,
  under `fade` it is a sell.
* **Volume panel** — the time-of-day expected volume and the `vol_mult`
  trigger; marked bars are sessions whose frozen first-K relative volume
  cleared the participation gate.

Remember that a symbol showing **no trades on a day it clearly gapped and
broke** is not a bug: it lost its seat on that morning's watchlist to a
higher-ranked name.

In [ ]:
def inspect_symbol(sym, pf, params):
    """Deep-dive one symbol of a portfolio: annotated chart + its trade table."""
    all_trades = bt.trade_records(pf)
    # high/low go in explicitly: signals() builds the opening range from the TRUE
    # high/low, so without them the drawn range is a close-based proxy and the chart
    # shows a level the signal never used.
    merged = {**strat.params, **params, "bar": panel.bar, "tz": panel.tz,
              "high": panel.high[sym], "low": panel.low[sym]}
    overlays = (strat.indicators(panel.close[sym], panel.volume[sym], **merged)
                if strat.indicators else None)
    plots.plot_symbol(panel.close[sym], panel.volume[sym], all_trades, symbol=sym,
                      overlays=overlays, equity=pf.value(group_by=False)[sym]).show()
    mine = all_trades[all_trades["symbol"] == sym]
    return mine.sort_values("entry_ts").reset_index(drop=True).round(3)

trades = bt.trade_records(pf)
busiest = trades["symbol"].value_counts().index[0]
inspect_symbol(busiest, pf, strat.params)

## 5 · Grid sweep — the ablation grid, mapped

Before optimizing, sweep the hand-picked grid from `config.yaml` — for this
strategy the grid IS the experiment: `polarity` × `entry_mode` × `gap_min` ×
`vol_mult` spans the thesis, its inverse, and both controls side by side, one
variable at a time. (`or_minutes` is deliberately *not* in the grid; range
length is a dial for the optimizer, not a thesis question.) The ranked bars
show the best single combos; the **heatmap grid** below them is the
multi-objective view — the same parameter pair judged by the annualized edge
over buy-and-hold (`excess_cagr`), total return *and* breadth
(`pct_profitable` = share of symbols that made money) at once, so a region
that pays on return but is carried by one name stands out immediately.

`chunk_size` is lowered here on purpose. `bt.sweep` column-stacks combos into
one vectorbt call, and the setting's default of 12 × 200 symbols is 2,400
stacked columns of 1-minute bars — enough to exhaust memory. Two combos per
call keeps the peak bounded; the result is identical either way.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
print(f"{len(grid)} combos")
sweep_stats = bt.sweep(strat, panel, grid, settings=settings, chunk_size=2)
sweep_stats.sort_values("excess_cagr", ascending=False).head(8).round(3)

In [ ]:
plots.plot_sweep(sweep_stats).show()
plots.plot_heatmap_grid(
    sweep_stats,
    pairs=[("gap_min", "polarity"), ("vol_mult", "entry_mode")],
    metrics=("excess_cagr", "total_return", "pct_profitable"),
).show()

## 5b · Event study — measure the signal *before* trusting the backtest

A backtest is a signal wrapped in stops, a wallet, costs and a benchmark. When
it comes out negative, none of those layers tells you *which* one is at fault.
So measure the naked event first, the way a cross-sectional strategy checks
rank IC before anything else.

The method: take the bars this config would actually fill on, compute the
forward return **open-to-open** from that fill, and subtract the
cross-sectional mean of every symbol's return over the identical bars. That
subtraction removes the market — what is left is the event's own information,
in basis points, signed the way the strategy is positioned (long entries as-is,
shorts sign-flipped). A positive number means the event predicts the position;
a negative one means the strategy would do better inverted.

Three readings to take:

* **Full thesis vs the gate-free control.** If gating on a gap makes the
  number *worse*, the gap is not a filter — it is the problem.
* **Capped vs uncapped.** The capped row is what the strategy trades (≤
  `max_candidates` per morning); the uncapped row is the signal's full
  population, with a much larger `n` and therefore a much more trustworthy
  t-stat. They should agree in sign; if the cap flips it, the watchlist
  ranking is doing the work rather than the event.
* **Fill delay.** A signal that only exists at delay 0 is bid-ask bounce, not
  information. Real effects survive being entered a few bars late — smaller,
  but the same sign.

`polarity: fade` is the exact mirror of these numbers, so a significantly
negative `go` row *is* the case for `fade`. It is not the verdict: an edge of a
few basis points has to survive a round trip that costs 6–46 bps (§12), and
this whole section is in-sample. §11 settles it.

In [ ]:
import numpy as np

HORIZONS = (15, 30, 60, 120)   # bars held, open-to-open from the fill bar
UNCAPPED = len(panel.symbols)  # watchlist wide enough to keep every candidate

def event_excess(params, *, delay=0):
    """Market-demeaned forward return of the events one config would fill on.

    Args:
        params: Parameter overrides on top of the config defaults.
        delay: Extra bars between the signal's fill bar and the measured entry
            — the microstructure control (an effect that survives a delay is
            information; one that vanishes was the spread).

    Returns:
        One row per horizon: event count, mean excess return in bps, and its
        t-stat. Signed as the strategy is positioned, so negative = the
        opposite `polarity` is the better trade on this evidence.
    """
    sig = strat.signals(panel, **{**strat.params, **params})
    long_, short = sig["entries"].to_numpy(), sig["short_entries"].to_numpy()
    rows = []
    for bars in HORIZONS:
        fwd = panel.open_.shift(-(bars + delay)) / panel.open_.shift(-delay) - 1.0
        # Demean across symbols at each bar: the market goes, the event stays.
        excess = fwd.sub(fwd.mean(axis=1), axis=0).to_numpy()
        vals = np.concatenate([excess[long_], -excess[short]])
        vals = vals[np.isfinite(vals)]
        rows.append({"bars": bars, "events": len(vals), "excess_bps": 1e4 * vals.mean(),
                     "t": vals.mean() / vals.std() * np.sqrt(len(vals))})
    return pd.DataFrame(rows).set_index("bars")

CONFIGS = {
    "full thesis (go)":       {},
    "full thesis, uncapped":  {"max_candidates": UNCAPPED},
    "gates off (any break)":  {"gap_min": 0.0, "vol_mult": 0.0, "max_candidates": UNCAPPED},
    "gap only (time entry)":  {"entry_mode": "time", "max_candidates": UNCAPPED},
    "big gaps only (4%)":     {"gap_min": 0.04, "max_candidates": UNCAPPED},
}
events = pd.concat({name: event_excess(p) for name, p in CONFIGS.items()}, names=["config"])
delayed = pd.concat({f"fill +{k} bars": event_excess({"max_candidates": UNCAPPED}, delay=k)
                     for k in (0, 2, 5)}, names=["control"])
print("Market-excess return of the event, signed as the strategy is positioned.")
print("Negative = polarity: fade is the better side of the same event.\n")
display(events.round(2))
delayed.round(2)

## 6 · Optimize (resumable, runs outside the notebook)

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study
maximizes **`info_ratio` and `excess_cagr` together** — both measured against
an equal-weight buy-and-hold of the same panel, so a trial scores above zero
only by *beating the market*, not by being long in a rising one; both
annualized, so windows of different lengths stay comparable. The result is a
*Pareto front* — the set of parameter sets where improving one objective must
hurt the other. Every finished trial is checkpointed to `studies/{study}.log`;
**re-running this cell resumes** exactly where it stopped (Ctrl-C / kernel
restarts lose nothing).

While it runs, every finished trial streams one line below — its objectives,
the current front size, and its parameters — so you can watch the search move.
For a richer live view, copy the **`uvx optuna-dashboard …`** command from the
first output line into a second terminal: it serves parameter importances,
objective history and the front at http://127.0.0.1:8080 while the study runs.

In [ ]:
N_TRIALS = 1000   # raise for a deeper search; re-runs accumulate into the same study
jobs.run_optimize("gap_and_go", n_trials=N_TRIALS)


## 7 · Read the study: Pareto front and the picked trial

Gray dots are all completed trials; the blue **staircase** is the exact
boundary of the dominated region (each labeled point is one Pareto-optimal
trial); the orange star is the trial `pick_best` selects under the weights
below. Hover any point for its full parameter set.

> **Read the prune rate first.** In v1, **90% of trials never scored** — 17%
> rejected by `valid()` before any backtest, 73% floor-pruned by
> `min_trades_per_year`, and the floor bit hardest on exactly the *selective*
> configs (the whole `gap_min ≥ 5%` region, everything at `vol_mult ≥ 3`).
> Both causes are addressed in v2 — relative parameterization for the first,
> a 200-name universe and a 100/yr floor for the second — but the check still
> belongs here. If the front again shows no selective configs, ask whether the
> floor is binding before concluding anything about big gaps.

The admissibility filter below re-applies the floor when reading the front.
`max_candidates` caps the achievable rate at `252 × max_candidates` per year,
so a concentrated watchlist and a high floor are in direct tension — if
nothing is admissible, that tension is the reason.

In [ ]:
study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)

# Prune accounting, because a Pareto front says nothing about the 90% of trials
# that never reached it. States: COMPLETE / PRUNED (invalid combo, no trades, or
# below the trade floor) / FAIL.
states = pd.Series([t.state.name for t in study.trials]).value_counts()
print(f"{len(study.trials)} trials — " + ", ".join(f"{n} {s.lower()}" for s, n in states.items()))

# Objectives are benchmark-relative and annualized: info_ratio (active return per unit of
# tracking risk) and excess_cagr (annualized edge over equal-weight buy-and-hold). Two
# guards before picking: (1) an admissibility floor on train trades — a handful of lucky
# fills can post a huge info_ratio that means nothing; (2) `opt.pick_from_front`, the ONE
# picking rule in the repo (opt.PICK_WEIGHTS, balanced), which collapses tied outcomes
# first so a coin flip between identical-scoring trials cannot masquerade as a decision.
# Watch `n_tied`: > 1 means the pick was arbitrary among that many equal strategies.
MIN_TRADES = 100      # admissibility floor for TRAIN trades — below this, ranking is noise.
                      # Was 300, which on the old 20-name universe was unreachable for any
                      # selective config; on 200 names it is comfortably clear of the floor.

eligible = front[front["n_trades"] >= MIN_TRADES]
if eligible.empty:
    print(f"!! no Pareto trial reaches {MIN_TRADES} train trades — showing the raw front; "
          "treat any pick as inadmissible")
    eligible = front
objs = list(opt.OBJECTIVES)          # downstream cells rank the runner-ups by objs[0]
best_row = opt.pick_from_front(eligible)
best = next(t for t in study.best_trials if t.number == int(best_row["trial"]))
print(f"front: {len(front)} trials, {len(eligible)} admissible (>= {MIN_TRADES} trades), "
      f"{int(best_row['front_distinct'])} distinct outcomes; picked trial {best.number}"
      + (f" — TIED with {int(best_row['n_tied']) - 1} other(s), so this pick is arbitrary"
         if best_row["n_tied"] > 1 else ""))
if "polarity" in front.columns:
    print("front by polarity:", front["polarity"].value_counts().to_dict(),
          f"— picked: {best.params.get('polarity')}")
plots.plot_pareto(front, everything, picked=best.number).show()
eligible.head(8).round(3)

## 8 · Boundary check — did the optimum hit the search-space edge?

A picked trial sitting **on the edge of a searched range** is a hint that the
true optimum may lie outside it. `optimize.boundary_report` compares the
picked parameters (and the whole Pareto front) with `search:` ranges from
`config.yaml`: any numeric parameter within 5% of a bound — measured on the
searched scale, so `log: true` ranges are judged in log space — is flagged
with the bound to widen. Categorical parameters have no edges and are skipped.

If something is flagged: widen that bound in `config.yaml`, re-run §6 (the
same study name resumes, and the sampler explores the new region; rename the
study to restart clean), then re-read §7–§8 until the front sits inside the
box. `tsim optimize` prints the same check at the end of every CLI run.

In [ ]:
report = opt.boundary_report(best.params, strat.optuna["space"], front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
report

## 9 · Re-run the full backtest with the picked parameters

In [ ]:
print("picked params:", best.params)
pf_best = bt.run_strategy(strat, panel, best.params, settings=settings)
stats_best = bt.portfolio_stats(pf_best)
stats_best |= bt.active_stats(pf_best, panel, fees=fees, slippage=slippage,
                              total_return=stats_best["total_return"])
pd.Series(stats_best).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf_best, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity with picked parameters").show()

## 10 · Verification — per-symbol deep dive

Portfolio means can hide an edge carried by a single name, so verify the
picked parameters symbol by symbol. First the full per-symbol breakdown of the
optimized backtest; then set `SYMBOL` in the next cell to any name that looks
interesting (the big winner, the one loser, the one with suspiciously few
trades) and re-run it to get the same annotated chart and trade table as §4 —
entries, exits, direction and the channel each breakout escaped.

In [ ]:
bt.symbol_stats(pf_best).round(4)

In [ ]:
SYMBOL = None   # ← set to any symbol from the table above (e.g. "NVDA"); None = most-traded
sym = SYMBOL or bt.trade_records(pf_best)["symbol"].value_counts().index[0]
inspect_symbol(sym, pf_best, best.params)

## 11 · Out-of-sample check — the tail the optimizer never saw

Everything above ran on data up to `data.end`, the **train cutoff**. This
section replays the picked parameters on the untouched tail — the one number
in this notebook the optimizer could not have fit.

The mechanics matter: signals are computed on the **full** history (so a
300-bar `trend_window` is warmed up on the day the holdout starts) and only
then are the panel and the signal frames sliced to the holdout rows, so every
metric below covers exclusively bars after the cutoff. Same symbols as the
train panel — re-resolving coverage over the holdout would quietly change the
universe. Expect worse numbers than §9; that is the point. Roughly comparable
Sharpe/breadth = the edge generalizes; a collapse to zero (or below) = §9 was
curve-fitting.

In [ ]:
from dataclasses import replace

assert d.get("end"), "set data.end in config.yaml to keep a holdout"
MIN_OOS_TRADES = 50
cutoff = pd.Timestamp(d["end"], tz="UTC")
full = load_panel(engine, panel.symbols, d["bar"], d["start"], None,
                  session=d.get("session", "regular"), exchanges=settings.exchanges)
after = full.close.index > cutoff
assert after.any(), "no bars stored after data.end — backfill the holdout tail first"
fees, slippage = bt.panel_costs(settings.costs, panel)
oos_panel = replace(full, close=full.close[after], open_=full.open_[after],
                    high=full.high[after], low=full.low[after], volume=full.volume[after])

def oos_portfolio(params):
    """Replay one parameter set on the untouched tail (warm-up sees history)."""
    sig = strat.signals(full, **{**strat.params, **params})
    oos_sig = {k: (v[after] if isinstance(v, pd.DataFrame) else v) for k, v in sig.items()}
    return bt.run(oos_panel, oos_sig, fees=fees, slippage=slippage,
                  init_cash=settings.backtest.init_cash)

def oos_stats(params):
    """Holdout stats incl. the benchmark-relative pair, on the holdout slice."""
    pf = oos_portfolio(params)
    st = bt.portfolio_stats(pf)
    return pf, st | bt.active_stats(pf, oos_panel, fees=fees, slippage=slippage,
                                    total_return=st["total_return"])

pf_oos, oos = oos_stats(best.params)
print(f"holdout: {full.close.index[after][0]:%Y-%m-%d} → {full.close.index[-1]:%Y-%m-%d} "
      f"({int(after.sum())} bars, {oos['n_trades']} trades)")
if oos["n_trades"] < MIN_OOS_TRADES:
    print(f"!! only {oos['n_trades']} OOS trades (< {MIN_OOS_TRADES}) — "
          "insufficient sample; do NOT read the metrics below as a verdict")
verdict = "BEATS" if oos["excess_cagr"] > 0 and oos["info_ratio"] > 0 else "does NOT beat"
print(f"hurdle: the pick {verdict} equal-weight buy-and-hold out of sample "
      f"(excess_cagr {oos['excess_cagr']:+.2%}, info_ratio {oos['info_ratio']:+.2f})")
plots.plot_equity(bt.equity_curve(pf_oos, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — OUT-OF-SAMPLE equity, picked parameters").show()

# Directional consistency: replay the two next-best admissible front rows.
# Params come from the FrozenTrials, NOT the frame rows — iterrows() would
# upcast int params (trend_window/horizon/...) to floats and rolling() raises.
by_number = {t.number: t for t in study.best_trials}
table = {"picked": oos}
for trial_no in eligible.sort_values(objs[0], ascending=False)["trial"]:
    if int(trial_no) == best.number or len(table) >= 3:
        continue
    table[f"trial {int(trial_no)}"] = oos_stats(by_number[int(trial_no)].params)[1]
pd.DataFrame({"train (§9)": stats_best,
              **{f"holdout · {k}": v for k, v in table.items()}}).round(4)

## 12 · Cost stress test — does the edge survive the open?

The single most important check for this strategy. The always-on cost model
charges the global `spread_bps: 2.0` — a liquid-large-cap *midday* number.
Quoted spreads in the first 15–45 minutes run **2–5× midday**, on top of the
volatile-name premium — and every entry this strategy will ever take lives in
that window. This cell re-prices the picked trades at two stress bars: the
**pass bar** (10 bps spread + 3 bps slippage — holdout `excess_cagr` must stay
positive here for the strategy to ship) and the **conviction bar** (20/5 —
opening-auction-adjacent worst case, reported either way). The benchmark is
re-priced at the same stressed costs, so the hurdle stays fair.

Put §5b's numbers next to these before reading anything as a result. The
event's market-excess return is a **single-digit basis-point** effect at these
horizons; a round trip costs ~6 bps at default costs and ~46 bps at the
conviction bar. An edge that only clears the bar at 2 bps has not been shown
to exist — it has been shown to be smaller than the spread it must cross.

Pair this with the **ablation** (§5/§7): if `entry_mode: time` or the
gates-off control matches the full thesis, the ORB trigger (or the gap) added
nothing — report that, don't ship it.

In [ ]:
# Env override for a CLI run: TRADING_COSTS__SPREAD_BPS=10 TRADING_COSTS__SLIPPAGE_BPS=3.
def stats_at(pf, f, s):
    st = bt.portfolio_stats(pf)
    return st | bt.active_stats(pf, panel, fees=f, slippage=s,
                                total_return=st["total_return"])

sig_best = strat.signals(panel, **{**strat.params, **best.params})
f_def, s_def = bt.panel_costs(settings.costs, panel)
columns = {"default costs (§9)": stats_at(pf_best, f_def, s_def)}
for label, spread, slip in [("pass bar 10/3 bps", 10.0, 3.0),
                            ("conviction bar 20/5 bps", 20.0, 5.0)]:
    stressed = settings.costs.model_copy(update={"spread_bps": spread, "slippage_bps": slip})
    f_str, s_str = bt.panel_costs(stressed, panel)
    pf_stress = bt.run(panel, sig_best, fees=f_str, slippage=s_str,
                       init_cash=settings.backtest.init_cash)
    columns[label] = stats_at(pf_stress, f_str, s_str)
    print(f"round-trip cost  {label:24s} ≈ {2 * (f_str + s_str) * 1e4:.1f} bps")
pd.DataFrame(columns).round(4)


## Read the result honestly

* §3–§10 are **in-sample**: the optimizer saw all the data they are judged on.
  §11 is the honest number — and even it gets stale after a few re-reads
  (each "check the holdout, tweak, check again" loop quietly fits the tail
  too). For a fresh view, re-run the pick on a *different universe*.
* **§5b outranks §9.** A backtest can be dragged positive by a lucky stop
  placement or one runaway morning; the event study measures the thing the
  strategy is actually betting on, over thousands of events, with the market
  removed. If they disagree, believe the event study and go looking for what
  in the execution layer produced the difference.
* Prefer combos where **`pct_profitable` is high** — on market-wide gap days
  the watchlist fills with one correlated morning bet, so an edge carried by
  one symbol is usually noise.
* A `polarity: fade` win is only interesting if it beats the `go` run *and*
  the gates-off control at stressed costs. "Fading gaps works" and "the gap
  gate is noise and any morning breakout mean-reverts" predict the same sign
  and very different strategies.
* The verdict logic is written down in the README: the shipped configuration
  must beat the `time` control, the gates-off control AND the baseline, on
  train and holdout, at stressed costs. Anything less ships as a finding, not
  a config.

**Add your own strategy:** copy `strategies/_template/`, write `signals()`, fill
in `config.yaml` — this exact notebook flow works unchanged. See the README's
"How to add a strategy".